<a href="https://colab.research.google.com/github/ShaneHurley/BasketballElo/blob/main/2026ELOcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [115]:
# -------------------------------
# 1. Imports & Setup
# -------------------------------
!pip install pandas==2.2.3 nbainjuries scikit-learn xgboost nba_api tqdm -q

import pandas as pd
import numpy as np
import warnings
from tqdm.auto import tqdm
from datetime import datetime
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import log_loss
from nbainjuries import injury

warnings.filterwarnings('ignore')

✅ Cell 1: Core environment and support packages initialized successfully.


In [116]:
# -------------------------------
# 2. Paths, Mappings, and Base Constants
# -------------------------------
from google.colab import drive
drive.mount('/drive', force_remount=True)

# File Paths
PLAYER_LIST_PATH = r"/drive/MyDrive/basketballData/BasketballELO/PlayerList08.csv"
PBP_PATH = r"/drive/MyDrive/basketballData/[10-21-2025]-[05-18-2026]-combined-stats.csv"
ODDS_CSV_PATH = r"/drive/MyDrive/basketballData/nba_main_lines.csv"

# Load Players
names_df = pd.read_csv(PLAYER_LIST_PATH, low_memory=False)
names_dict = names_df.set_index('PERSON_ID').to_dict()['DISPLAY_FIRST_LAST']
name_to_id = {v: k for k, v in names_dict.items()}

# Team Dictionary
team_name_to_abbr = {
    'Atlanta Hawks': 'ATL', 'Boston Celtics': 'BOS', 'Brooklyn Nets': 'BKN',
    'Charlotte Hornets': 'CHA', 'Chicago Bulls': 'CHI', 'Cleveland Cavaliers': 'CLE',
    'Dallas Mavericks': 'DAL', 'Denver Nuggets': 'DEN', 'Detroit Pistons': 'DET',
    'Golden State Warriors': 'GSW', 'Houston Rockets': 'HOU', 'Indiana Pacers': 'IND',
    'LA Clippers': 'LAC', 'Los Angeles Clippers': 'LAC', 'Los Angeles Lakers': 'LAL',
    'Memphis Grizzlies': 'MEM', 'Miami Heat': 'MIA', 'Milwaukee Bucks': 'MIL',
    'Minnesota Timberwolves': 'MIN', 'New Orleans Pelicans': 'NOP', 'New York Knicks': 'NYK',
    'Oklahoma City Thunder': 'OKC', 'Orlando Magic': 'ORL', 'Philadelphia 76ers': 'PHI',
    'Phoenix Suns': 'PHX', 'Portland Trail Blazers': 'POR', 'Sacramento Kings': 'SAC',
    'San Antonio Spurs': 'SAS', 'Toronto Raptors': 'TOR', 'Utah Jazz': 'UTA',
    'Washington Wizards': 'WAS'
}

# Base constants that don't need calibration
BASE_ELO = 1500
ASSIST_SPLIT = 0.65
USAGE_FLOOR = 0.15

Mounted at /drive
Loading data from Google Drive...
✅ Cell 2: Drive Mounted. Loaded 4393 players, 753077 PBP rows, and 8129 Odds rows.


In [125]:
# -------------------------------
# 3. Conversion and Data-Driven xPoints
# -------------------------------
import pandas as pd
import numpy as np

def convert_new_pbp_to_legacy(pbp_df, name_to_id):
    df = pbp_df.copy()
    df['PERIOD'] = df.get('period', 1).astype(int)
    df['GAME_ID'] = df['game_id']
    df['HOME_SCORE'] = df['home_score']
    df['AWAY_SCORE'] = df['away_score']
    df['type'] = df.get('type', '')

    conditions = [
        (df['event_type'] == 'shot') & (df['result'] == 'made'),
        (df['event_type'] == 'shot') & (df['result'] == 'missed'),
        (df['event_type'] == 'free throw'),
        (df['event_type'] == 'rebound'),
        (df['event_type'] == 'turnover'),
        (df['event_type'] == 'foul'),
        (df['event_type'] == 'substitution')
    ]
    choices = [1, 2, 3, 4, 5, 6, 8]
    df['EVENTMSGTYPE'] = np.select(conditions, choices, default=0)
    df['points'] = df['points'].fillna(0).astype(int)

    df['HOMEDESCRIPTION'] = np.where(df['team'] == df['home_team'], df['description'], np.nan)
    df['VISITORDESCRIPTION'] = np.where(df['team'] == df['away_team'], df['description'], np.nan)

    def map_name(name):
        if pd.isna(name) or name == '': return None
        s_name = str(name)
        if '/' in s_name:
            parts = s_name.split('/')
            try: return int(parts[0])
            except ValueError: s_name = parts[-1]
        pid = name_to_id.get(s_name)
        return int(pid) if pid is not None else hash(s_name) % 1000000000

    df['PLAYER1_ID'] = df['player'].apply(map_name)
    df['PLAYER2_ID'] = df['assist'].apply(map_name)
    df['PLAYER3_ID'] = df['block'].apply(map_name)
    df['PLAYER1_TEAM_ID'] = df['team']

    for i in range(1, 6):
        if f'h{i}' in df.columns: df[f'h{i}_id'] = df[f'h{i}'].apply(map_name)
        if f'a{i}' in df.columns: df[f'a{i}_id'] = df[f'a{i}'].apply(map_name)

    h_cols = [f'h{i}_id' for i in range(1,6) if f'h{i}_id' in df.columns]
    a_cols = [f'a{i}_id' for i in range(1,6) if f'a{i}_id' in df.columns]

    home_arr = df[h_cols].values
    away_arr = df[a_cols].values

    df['HOME_players'] = ['-'.join(map(str, sorted(int(x) for x in row if pd.notna(x)))) for row in home_arr]
    df['AWAY_players'] = ['-'.join(map(str, sorted(int(x) for x in row if pd.notna(x)))) for row in away_arr]

    df = df[(df['HOME_players'] != '') & (df['AWAY_players'] != '')].reset_index(drop=True)
    return df

def preprocess_pbp(pbp_df):
    df = pbp_df.copy()
    is_home = df['HOMEDESCRIPTION'].notna() & df['VISITORDESCRIPTION'].isna()
    is_away = df['VISITORDESCRIPTION'].notna() & df['HOMEDESCRIPTION'].isna()
    is_shot = df['EVENTMSGTYPE'].isin([1, 2])
    is_fg_make = (df['EVENTMSGTYPE'] == 1)
    is_fg_miss = (df['EVENTMSGTYPE'] == 2)
    is_tov = (df['EVENTMSGTYPE'] == 5)

    df['is_fg_make'] = is_fg_make
    df['is_fg_miss'] = is_fg_miss
    df['is_tov'] = is_tov

    # ---- 1. Assign shot zones ----
    dist = df.get('shot_distance', pd.Series([-1]*len(df))).fillna(-1)

    # Detect three‑pointers: text '3pt' OR distance >= 23.75
    is_three = df['type'].str.contains('3pt', case=False, na=False, regex=False)
    if is_three.sum() == 0:
        is_three = (dist >= 23.75) & (dist < 30)
        print(f"Falling back to distance‑based three‑point detection: {is_three.sum()} attempts")

    # No corner distinction – all threes become 'ab3'
    is_rim = df['type'].str.contains('Layup|Dunk|Tip', case=False, na=False) | ((dist <= 3) & (dist >= 0))
    is_short_mid = (dist > 3) & (dist <= 10)
    is_long_mid = (dist > 10) & (dist < 23.75)

    zone = np.select(
        [is_three, is_rim, is_short_mid, is_long_mid],
        ['ab3', 'rim', 'short_mid', 'long_mid'],
        default='other'
    )
    df['shot_zone'] = zone

    # ---- 2. Compute actual points per attempt for each zone ----
    shot_make_value = np.where(is_three, 3, 2)
    actual_points = np.where(is_fg_make, shot_make_value, 0)

    zone_stats = {}
    for z in ['ab3', 'rim', 'short_mid', 'long_mid']:
        mask = (df['shot_zone'] == z) & is_shot
        attempts = mask.sum()
        if attempts > 0:
            points = actual_points[mask].sum()
            makes = is_fg_make[mask].sum()
            pps = points / attempts
            zone_stats[z] = {'attempts': attempts, 'makes': makes, 'pps': pps}
        else:
            zone_stats[z] = {'attempts': 0, 'makes': 0, 'pps': 1.0}

    print("\n=== Data‑driven Points Per Shot Attempt ===")
    for z, stats in zone_stats.items():
        print(f"{z:10s}: {stats['pps']:.3f}  (attempts: {stats['attempts']}, makes: {stats['makes']})")

    zone_to_xpts = {z: stats['pps'] for z, stats in zone_stats.items()}
    zone_to_xpts['other'] = 0.90

    df['xPoints'] = df['shot_zone'].map(zone_to_xpts)
    df.loc[~is_shot, 'xPoints'] = 0.0

    # ---- 3. Points added (actual) ----
    df['home_pts_added'] = 0
    df['away_pts_added'] = 0

    make_home = is_fg_make & is_home
    df.loc[make_home, 'home_pts_added'] = shot_make_value[make_home]
    make_away = is_fg_make & is_away
    df.loc[make_away, 'away_pts_added'] = shot_make_value[make_away]

    # Free throws
    ft_made = (df['EVENTMSGTYPE'] == 3) & (~df['result'].str.contains('missed', na=False))
    df.loc[ft_made & is_home, 'home_pts_added'] = 1
    df.loc[ft_made & is_away, 'away_pts_added'] = 1

    # ---- 4. Expected points added (using xPoints) ----
    df['home_xpts_added'] = 0.0
    df['away_xpts_added'] = 0.0
    df.loc[is_shot & is_home, 'home_xpts_added'] = df.loc[is_shot & is_home, 'xPoints']
    df.loc[is_shot & is_away, 'away_xpts_added'] = df.loc[is_shot & is_away, 'xPoints']

    ft_attempt = (df['EVENTMSGTYPE'] == 3)
    ft_make_pct = ft_made.sum() / ft_attempt.sum() if ft_attempt.sum() > 0 else 0.77
    df.loc[ft_attempt & is_home, 'home_xpts_added'] = ft_make_pct
    df.loc[ft_attempt & is_away, 'away_xpts_added'] = ft_make_pct
    df['is_true_ft_trip'] = (df['EVENTMSGTYPE'] == 3) & (df['type'].str.contains('1 of 1|2 of 2|3 of 3', regex=True, na=False))

    # ---- 5. Possessions ----
    df['home_poss'] = ((df['EVENTMSGTYPE'].isin([1,2,5])) * is_home.astype(int))
    df['away_poss'] = ((df['EVENTMSGTYPE'].isin([1,2,5])) * is_away.astype(int))
    df.loc[df['is_true_ft_trip'] & is_home, 'home_poss'] += 0.44
    df.loc[df['is_true_ft_trip'] & is_away, 'away_poss'] += 0.44
    df['total_possessions'] = (df['home_poss'] + df['away_poss']) / 2

    # ---- 6. Garbage time ----
    def extract_minutes(t):
        if pd.isna(t): return 12.0
        parts = str(t).split(':')
        if len(parts) == 2:
            return int(parts[0]) + int(parts[1])/60.0
        return 12.0
    df['min_rem'] = df['remaining_time'].apply(extract_minutes) if 'remaining_time' in df.columns else 12.0
    df['margin'] = abs(df['HOME_SCORE'] - df['AWAY_SCORE'])
    df['garbage'] = (df['PERIOD'] == 4) & (
        (df['margin'] >= 20) |
        ((df['margin'] >= 15) & (df['min_rem'] <= 3.0))
    )

    garbage_cols = ['total_possessions', 'home_pts_added', 'away_pts_added', 'home_xpts_added', 'away_xpts_added']
    df.loc[df['garbage'], garbage_cols] = 0

    return df

In [126]:
# -------------------------------
# 4. Building Stints & Games
# -------------------------------
def build_stints_and_usage(pbp_df):
    # Create stint ID
    pbp_df['stint_id'] = (
        (pbp_df['HOME_players'] != pbp_df['HOME_players'].shift()) |
        (pbp_df['AWAY_players'] != pbp_df['AWAY_players'].shift()) |
        (pbp_df['PERIOD'] != pbp_df['PERIOD'].shift())
    ).cumsum()

    # Aggregate only columns that exist
    stints_df = pbp_df.groupby('stint_id').agg(
        GAME_ID=('GAME_ID', 'first'),
        PERIOD=('PERIOD', 'first'),
        HOME_players=('HOME_players', 'first'),
        AWAY_players=('AWAY_players', 'first'),
        HOME_SCORE_START=('HOME_SCORE', 'first'),
        AWAY_SCORE_START=('AWAY_SCORE', 'first'),
        home_pts=('home_pts_added', 'sum'),
        away_pts=('away_pts_added', 'sum'),
        home_xpts=('home_xpts_added', 'sum'),
        away_xpts=('away_xpts_added', 'sum'),
        possessions=('total_possessions', 'sum')
    ).reset_index()

    stints_df['HOME_SCORE_END'] = stints_df['HOME_SCORE_START'] + stints_df['home_pts']
    stints_df['AWAY_SCORE_END'] = stints_df['AWAY_SCORE_START'] + stints_df['away_pts']

    # ---- Usage calculation ----
    home_u, away_u = {}, {}
    for _, row in stints_df.iterrows():
        sid = row['stint_id']
        home_u[sid] = {p: 0.0 for p in row['HOME_players'].split('-') if p and p != 'nan'}
        away_u[sid] = {p: 0.0 for p in row['AWAY_players'].split('-') if p and p != 'nan'}

    records = pbp_df.to_dict('records')
    for i, row in enumerate(records):
        stint = row['stint_id']
        p1 = str(int(row['PLAYER1_ID'])) if pd.notna(row.get('PLAYER1_ID')) else None
        p2 = str(int(row['PLAYER2_ID'])) if pd.notna(row.get('PLAYER2_ID')) else None

        is_make = row.get('is_fg_make', False)
        is_miss = row.get('is_fg_miss', False)
        is_tov = row.get('is_tov', False)
        is_true_ft_trip = row.get('is_true_ft_trip', False)

        # Detect if next event is an offensive rebound by same team (to avoid crediting usage on missed FG that is rebounded)
        next_is_oreb = False
        if is_miss and i + 1 < len(records):
            nxt = records[i+1]
            if nxt.get('EVENTMSGTYPE') == 4 and nxt.get('PLAYER1_TEAM_ID') == row.get('PLAYER1_TEAM_ID'):
                next_is_oreb = True

        hu = home_u.get(stint, {})
        au = away_u.get(stint, {})

        if is_make:
            if p2 in hu and p1 in hu:
                hu[p1] += ASSIST_SPLIT
                hu[p2] += (1.0 - ASSIST_SPLIT)
            elif p1 in hu:
                hu[p1] += 1.0
            if p2 in au and p1 in au:
                au[p1] += ASSIST_SPLIT
                au[p2] += (1.0 - ASSIST_SPLIT)
            elif p1 in au:
                au[p1] += 1.0
        elif is_miss and not next_is_oreb:
            if p1 in hu:
                hu[p1] += 1.0
            if p1 in au:
                au[p1] += 1.0
        elif is_tov or is_true_ft_trip:
            if p1 in hu:
                hu[p1] += 1.0
            if p1 in au:
                au[p1] += 1.0

    stints_df['home_usage'] = stints_df['stint_id'].map(home_u)
    stints_df['away_usage'] = stints_df['stint_id'].map(away_u)
    # For defensive usage, we simply reuse offensive usage (or could set to zeros – not critical for new tracker)
    stints_df['home_def_usage'] = stints_df['home_usage']
    stints_df['away_def_usage'] = stints_df['away_usage']

    # Filter out garbage stints (possessions == 0)
    return stints_df[stints_df['possessions'] > 0].reset_index(drop=True)

def prepare_game_outcomes(stints_df, raw_pbp_df):
    games = {}
    for _, s in stints_df.iterrows():
        gid = s['GAME_ID']
        if gid not in games:
            team_info = raw_pbp_df[raw_pbp_df['game_id'] == gid].iloc[0]
            games[gid] = {
                'home_score': 0, 'away_score': 0,
                'home_team': team_info['home_team'], 'away_team': team_info['away_team'],
                'game_date': team_info.get('date', None)
            }
        games[gid]['home_score'] += s['home_pts']
        games[gid]['away_score'] += s['away_pts']

    records = []
    for gid, data in games.items():
        records.append({
            'GAME_ID': gid, 'game_date': pd.to_datetime(data['game_date']).date(),
            'home_score': data['home_score'], 'away_score': data['away_score'],
            'home_win': 1 if data['home_score'] > data['away_score'] else 0,
            'home_team': data['home_team'], 'away_team': data['away_team']
        })
    df = pd.DataFrame(records)
    return df.sort_values('game_date').reset_index(drop=True)

In [127]:
# -------------------------------
# 5. Process-Based Elo Tracker (xPPP & Time Decay)
# -------------------------------
import datetime as dt
import numpy as np

class PlayerRatingTracker:
    def __init__(self, config, weights):
        self.players = {}
        self.config = config
        self.weights = weights
        self.base_elo = BASE_ELO

    def get_player(self, pid):
        pid = str(pid)
        if pid not in self.players:
            self.players[pid] = {
                'O_Elo': self.base_elo,
                'D_Elo': self.base_elo,
                'Possessions': 0,
                'last_game_date': None
            }
        return self.players[pid]

    def apply_inactivity_decay(self, active_player_ids, current_date):
        for pid in active_player_ids:
            p = self.get_player(pid)
            if p['last_game_date'] is not None:
                days_inactive = (current_date - p['last_game_date']).days
                if days_inactive > 60:
                    # Regress towards 1500 by a maximum of 25% of the difference
                    o_diff = p['O_Elo'] - self.base_elo
                    d_diff = p['D_Elo'] - self.base_elo
                    p['O_Elo'] -= (o_diff * 0.25)
                    p['D_Elo'] -= (d_diff * 0.25)
            p['last_game_date'] = current_date

    def get_lineup_rating(self, player_ids):
        active_ids = [str(pid) for pid in player_ids if pid and str(pid) != 'nan']
        n = len(active_ids)
        base_w = 1.0 / n if n > 0 else 0.2
        o_sum = sum(self.get_player(pid)['O_Elo'] for pid in active_ids)
        d_sum = sum(self.get_player(pid)['D_Elo'] for pid in active_ids)
        avg_exp = np.mean([self.get_player(pid)['Possessions'] for pid in active_ids]) if active_ids else 0
        return {'O_Elo': o_sum * base_w, 'D_Elo': d_sum * base_w, 'Avg_Exp': avg_exp, 'Base_Weight': base_w}, [base_w]*n

    def update_ratings(self, player_ids, o_shift, d_shift, poss, off_usage, def_usage, base_w):
        active_ids = [str(pid) for pid in player_ids if pid and str(pid) != 'nan']
        if not active_ids: return

        total_o = sum(off_usage.values())
        total_d = sum(def_usage.values())
        o_shares, d_shares = [], []

        for pid in active_ids:
            raw_o = (off_usage.get(pid, 0) / total_o) if total_o > 0 else base_w
            o_shares.append(max(raw_o, base_w * USAGE_FLOOR))
            raw_d = (def_usage.get(pid, 0) / total_d) if total_d > 0 else base_w
            d_shares.append((0.80 * base_w) + (0.20 * raw_d))

        sum_o, sum_d = sum(o_shares), sum(d_shares)
        for i, pid in enumerate(active_ids):
            p = self.get_player(pid)
            p['O_Elo'] += (o_shift * 0.90 * ((o_shares[i]/sum_o) / base_w))
            p['D_Elo'] += (d_shift * 0.90 * ((d_shares[i]/sum_d) / base_w))
            p['Possessions'] += poss

    def process_stint(self, ids_A, ids_B, poss, xpts_A, xpts_B, usage_A, usage_B,
                      period, start_score_A, start_score_B, end_score_A, end_score_B, season_progress=0.5):
        """
        ids_A, ids_B: lists of player IDs (strings)
        poss: number of possessions in this stint
        xpts_A, xpts_B: total expected points scored by each lineup
        usage_A, usage_B: dicts {player_id: usage_share}
        """
        if poss <= 0:
            return 0.0, 0.0

        rating_A, _ = self.get_lineup_rating(ids_A)
        rating_B, _ = self.get_lineup_rating(ids_B)

        # Predicted Points Per Possession (PPP)
        # LEAGUE_XPPP must be around 1.1
        pred_ppp_A = self.weights['LEAGUE_XPPP'] + self.config.get('HOME_PPP_BOOST', 0.0) + \
                     ((rating_A['O_Elo'] - rating_B['D_Elo']) / self.config['ELO_SCALING_FACTOR'])
        pred_ppp_B = self.weights['LEAGUE_XPPP'] - self.config.get('HOME_PPP_BOOST', 0.0) + \
                     ((rating_B['O_Elo'] - rating_A['D_Elo']) / self.config['ELO_SCALING_FACTOR'])

        # Residual = Actual PPP - Predicted PPP
        res_A = (xpts_A / poss) - pred_ppp_A
        res_B = (xpts_B / poss) - pred_ppp_B

        # K-factor with game situation adjustments
        avg_exp = min(rating_A['Avg_Exp'], rating_B['Avg_Exp'])
        k_mult = np.clip(avg_exp / 500.0, 0.2, 1.0)
        lead = abs(end_score_A - end_score_B)
        leverage = 2.0 if period > 4 else np.clip(0.5 + 1.5 * np.exp(-0.05 * lead) * (period / 4.0), 0.3, 2.0)
        chaos_damp = 0.7 if poss > 25 else (0.9 if poss > 18 else 1.0)
        recency_amp = 1.0 + (0.20 * season_progress)

        k_off = self.config['K_OFF'] * k_mult * leverage * chaos_damp * recency_amp
        k_def = self.config['K_DEF'] * k_mult * leverage * chaos_damp * recency_amp

        # Update ratings – using usage shares to distribute credit
        self.update_ratings(ids_A, k_off * res_A, k_def * (-res_B), poss,
                            usage_A, usage_A, rating_A['Base_Weight'])
        self.update_ratings(ids_B, k_off * res_B, k_def * (-res_A), poss,
                            usage_B, usage_B, rating_B['Base_Weight'])

        return res_A, res_B

In [128]:
# -------------------------------
# 6. Extraction and Market Helpers
# -------------------------------
def prepare_odds_dict(odds_path, team_name_to_abbr):
    odds_df = pd.read_csv(odds_path)
    odds_df['timestamp'] = pd.to_datetime(odds_df['timestamp'])
    odds_df = odds_df.sort_values('timestamp', ascending=False)

    odds_dict = {}
    for _, row in odds_df.iterrows():
        t1_abbr = team_name_to_abbr.get(row['team1'])
        t2_abbr = team_name_to_abbr.get(row['team2'])
        if not t1_abbr or not t2_abbr: continue

        key = tuple(sorted([t1_abbr, t2_abbr]))
        if key not in odds_dict:
            odds_dict[key] = {
                'team1_abbr': t1_abbr, 'team2_abbr': t2_abbr,
                'team1_spread': row['team1_spread'], 'team2_spread': row['team2_spread'],
                'team1_moneyline': row['team1_moneyline'], 'team2_moneyline': row['team2_moneyline']
            }
    return odds_dict

def get_odds_for_game(home_abbr, away_abbr, odds_dict):
    key = tuple(sorted([home_abbr, away_abbr]))
    if key not in odds_dict: return None, None

    odds = odds_dict[key]
    if odds['team1_abbr'] == home_abbr:
        return odds['team1_spread'], odds['team1_moneyline']
    elif odds['team2_abbr'] == home_abbr:
        return odds['team2_spread'], odds['team2_moneyline']
    return None, None

def implied_prob(odds):
    if pd.isna(odds) or odds is None: return 0.5
    try:
        odds = float(odds)
        if 1.0 < odds < 50.0: return 1 / odds
        elif odds < 0: return -odds / (-odds + 100)
        else: return 100 / (odds + 100)
    except: return 0.5

print("Loading and mapping betting odds CSV...")
odds_dict = prepare_odds_dict(ODDS_CSV_PATH, team_name_to_abbr)

print("Loading new PBP data...")
new_pbp = pd.read_csv(PBP_PATH, low_memory=False)
legacy_pbp = convert_new_pbp_to_legacy(new_pbp, name_to_id)
legacy_pbp = preprocess_pbp(legacy_pbp)
stints_df = build_stints_and_usage(legacy_pbp)
game_outcomes_df = prepare_game_outcomes(stints_df, new_pbp)

total_xpts = stints_df['home_xpts'].sum() + stints_df['away_xpts'].sum()
total_poss = stints_df['possessions'].sum() * 2
LEAGUE_XPPP = total_xpts / total_poss if total_poss > 0 else 1.10

print(f"League average xPPP: {LEAGUE_XPPP:.3f}")

DYNAMIC_WEIGHTS = {
    'LEAGUE_XPPP': LEAGUE_XPPP,
    # Other weights are no longer used in process_stint, but kept for compatibility
    'WEIGHT_OREB': 0, 'WEIGHT_TOV': 0, 'WEIGHT_FOUL_DRAWN': 0, 'WEIGHT_BLK': 0
}
print(f"Generated Dynamic Weights: {DYNAMIC_WEIGHTS}")


Loading and mapping betting odds CSV...
Loading new PBP data...
Falling back to distance‑based three‑point detection: 75821 attempts

=== Data‑driven Points Per Shot Attempt ===
ab3       : 1.064  (attempts: 75819, makes: 26880)
rim       : 1.256  (attempts: 77415, makes: 48615)
short_mid : 0.923  (attempts: 25249, makes: 11652)
long_mid  : 0.829  (attempts: 51856, makes: 21499)
League average xPPP: 1.029
Generated Dynamic Weights: {'LEAGUE_XPPP': np.float64(1.029190230893348), 'WEIGHT_OREB': 0, 'WEIGHT_TOV': 0, 'WEIGHT_FOUL_DRAWN': 0, 'WEIGHT_BLK': 0}


In [129]:
def calibrate_hyperparameters(game_outcomes_df, stints_df, param_grid):
    from sklearn.metrics import log_loss
    from sklearn.model_selection import ParameterGrid
    from tqdm.auto import tqdm

    grid = list(ParameterGrid(param_grid))
    best_logloss = float('inf')
    best_params, best_tracker = None, None

    print(f"Starting calibration over {len(grid)} combinations...")

    for params in tqdm(grid, desc="Calibrating Variables"):
        tracker = PlayerRatingTracker(config=params, weights=DYNAMIC_WEIGHTS)
        y_true, y_pred = [], []

        for idx, game in game_outcomes_df.iterrows():
            gid = game['GAME_ID']
            stints_for_game = stints_df[stints_df['GAME_ID'] == gid]

            # Collect unique players for pre-game Elo averaging
            home_players = set()
            away_players = set()
            for _, st in stints_for_game.iterrows():
                home_players.update(st['HOME_players'].split('-'))
                away_players.update(st['AWAY_players'].split('-'))
            home_players = [p for p in home_players if p and p != 'nan']
            away_players = [p for p in away_players if p and p != 'nan']

            if home_players and away_players:
                home_off = np.mean([tracker.get_player(p)['O_Elo'] for p in home_players])
                home_def = np.mean([tracker.get_player(p)['D_Elo'] for p in home_players])
                away_off = np.mean([tracker.get_player(p)['O_Elo'] for p in away_players])
                away_def = np.mean([tracker.get_player(p)['D_Elo'] for p in away_players])

                pred_h = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] + params.get('HOME_PPP_BOOST', 0.0) + \
                         (home_off - away_def) / params['ELO_SCALING_FACTOR']
                pred_a = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] - params.get('HOME_PPP_BOOST', 0.0) + \
                         (away_off - home_def) / params['ELO_SCALING_FACTOR']

                model_spread = (pred_h - pred_a) * 100.0
                model_win_prob = 1 / (1 + 10 ** (-model_spread / 15.0))

                y_pred.append(model_win_prob)
                y_true.append(game['home_win'])

            # Update Elo using the NEW process_stint signature
            season_progress = idx / len(game_outcomes_df)
            for _, st in stints_for_game.iterrows():
                tracker.process_stint(
                    ids_A=st['HOME_players'].split('-'),
                    ids_B=st['AWAY_players'].split('-'),
                    poss=st['possessions'],
                    xpts_A=st['home_xpts'],
                    xpts_B=st['away_xpts'],
                    usage_A=st['home_usage'],
                    usage_B=st['away_usage'],
                    period=st['PERIOD'],
                    start_score_A=st['HOME_SCORE_START'],
                    start_score_B=st['AWAY_SCORE_START'],
                    end_score_A=st['HOME_SCORE_END'],
                    end_score_B=st['AWAY_SCORE_END'],
                    season_progress=season_progress
                )

        if len(y_pred) > 0:
            current_logloss = log_loss(y_true, y_pred)
            if current_logloss < best_logloss:
                best_logloss = current_logloss
                best_params = params
                best_tracker = tracker

    print(f"\n✅ Calibration Complete! Log-Loss: {best_logloss:.4f}")
    print(f"🌟 Optimal Config: {best_params}")
    return best_params, best_tracker

In [130]:
# -------------------------------
# 8. Run Model & Output Results
# -------------------------------
# Variables you want the Grid Search to test. Add or subtract values as needed!
param_grid = {
    'K_OFF': [0.25, 0.30, 0.35],
    'K_DEF': [0.65, 0.70, 0.75],
    'ELO_SCALING_FACTOR': [800, 1000, 1200, 1500],
    'HOME_PPP_BOOST': [0.020, 0.024]
}

optimal_params, optimal_tracker = calibrate_hyperparameters(game_outcomes_df, stints_df, param_grid)

results = []
for idx, game in game_outcomes_df.iterrows():
    gid = game['GAME_ID']
    home_abbr = game['home_team']
    away_abbr = game['away_team']
    game_date = game['game_date']

    # Safely query odds using abbreviation logic
    market_spread, market_moneyline = get_odds_for_game(home_abbr, away_abbr, odds_dict)
    implied = implied_prob(market_moneyline) if market_moneyline is not None else None

    # Injuries (Can add tqdm here if scraping slow)
    home_inj = [] # Replace with `fetch_injuries(home_abbr, game_date)` if needed
    away_inj = [] # Replace with `fetch_injuries(away_abbr, game_date)` if needed

    stints_for_game = stints_df[stints_df['GAME_ID'] == gid]
    home_players = set(p for st in stints_for_game['HOME_players'] for p in st.split('-') if p and p != 'nan')
    away_players = set(p for st in stints_for_game['AWAY_players'] for p in st.split('-') if p and p != 'nan')

    if home_players and away_players:
        home_off = np.mean([optimal_tracker.get_player(p)['O_Elo'] for p in home_players])
        home_def = np.mean([optimal_tracker.get_player(p)['D_Elo'] for p in home_players])
        away_off = np.mean([optimal_tracker.get_player(p)['O_Elo'] for p in away_players])
        away_def = np.mean([optimal_tracker.get_player(p)['D_Elo'] for p in away_players])

        pred_h = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] + optimal_params['HOME_PPP_BOOST'] + ((home_off - away_def) / optimal_params['ELO_SCALING_FACTOR'])
        pred_a = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] - optimal_params['HOME_PPP_BOOST'] + ((away_off - home_def) / optimal_params['ELO_SCALING_FACTOR'])

        model_spread = (pred_h - pred_a) * 100.0
        model_win_prob = 1 / (1 + 10 ** (-model_spread / 15.0))

        prob_edge = model_win_prob - implied if implied else None

        results.append({
            'Date': game_date, 'Home': home_abbr, 'Away': away_abbr,
            'Model_Spread': round(model_spread, 2), 'Market_Spread': market_spread,
            'Model_Win_Prob': round(model_win_prob, 4), 'Implied_Prob': round(implied, 4) if implied else None,
            'Prob_Edge': round(prob_edge, 4) if prob_edge else None,
            'Bet': ("Home" if prob_edge > 0.02 else "Away" if prob_edge < -0.02 else "No bet") if prob_edge else "N/A"
        })

print("\n=== Best Betting Opportunities ===")
results_df = pd.DataFrame(results)
if not results_df.empty:
    edges = results_df.dropna(subset=['Prob_Edge']).sort_values('Prob_Edge', key=abs, ascending=False)
    print(edges.head(15).to_string(index=False))

    output_path = "/drive/MyDrive/betting_predictions_2025_26_calibrated.csv"
    results_df.to_csv(output_path, index=False)
    print(f"\nDetailed predictions saved to {output_path}")

Starting calibration over 72 combinations...


Calibrating Variables:   0%|          | 0/72 [00:00<?, ?it/s]


✅ Calibration Complete! Log-Loss: 0.6998
🌟 Optimal Config: {'ELO_SCALING_FACTOR': 800, 'HOME_PPP_BOOST': 0.02, 'K_DEF': 0.65, 'K_OFF': 0.35}

=== Best Betting Opportunities ===
      Date Home Away  Model_Spread  Market_Spread  Model_Win_Prob  Implied_Prob  Prob_Edge  Bet
2025-11-24  MEM  DEN          5.01           23.0          0.6834        0.0410     0.6424 Home
2025-10-29  BKN  ATL          7.02           15.5          0.7460        0.1041     0.6419 Home
2026-01-05  PHI  DEN          6.65           16.0          0.7353        0.1041     0.6312 Home
2025-10-27  UTA  PHX          5.77           16.5          0.7081        0.0903     0.6178 Home
2026-01-21  MEM  ATL          6.32           14.5          0.7252        0.1176     0.6076 Home
2026-01-06  MEM  SAS          5.29           17.0          0.6926        0.0882     0.6044 Home
2026-03-18  MEM  DEN          3.83           23.0          0.6430        0.0410     0.6020 Home
2026-03-29  MIL  LAC          5.53           17.0     

In [131]:
# -------------------------------
# 9. Real Injury Fetching
# -------------------------------
import datetime as dt

def fetch_injuries(team_abbr, game_date, use_real=True):
    """
    Fetch list of injured players for a team on a given date.
    Returns a list of player names.
    """
    if not use_real:
        return []
    try:
        if isinstance(game_date, str):
            game_date = pd.to_datetime(game_date).date()
        elif isinstance(game_date, pd.Timestamp):
            game_date = game_date.date()
        elif isinstance(game_date, dt.datetime):
            game_date = game_date.date()

        df = injury.get_reportdata(game_date, return_df=True)
        team_df = df[df['team'].str.upper() == team_abbr.upper()]
        injured = team_df[team_df['status'].isin(['Out', 'Questionable'])]['player_name'].tolist()
        return injured
    except Exception as e:
        print(f"⚠️ Injury fetch failed for {team_abbr} on {game_date}: {e}")
        return []

In [132]:
# -------------------------------
# 10. ATS Backtest & Confidence Metrics
# -------------------------------
from sklearn.metrics import mean_absolute_error, brier_score_loss
from tqdm.auto import tqdm
import pandas as pd
import numpy as np

def calculate_bet_confidence(model_spread, market_home_spread, min_edge=2.5):
    """Returns dict with edge_points, bet, confidence_score, stars"""
    if pd.isna(market_home_spread) or market_home_spread is None:
        return {"edge_points": 0, "bet": "N/A", "confidence_score": 0, "stars": "N/A"}

    # Edge = model margin relative to market spread
    # If market spread is HOU -3, and model says HOU by 6 => edge = +3
    edge_points = model_spread + market_home_spread   # because market_home_spread is home team's spread (e.g., -3)
    abs_edge = abs(edge_points)
    bet = "Pass" if abs_edge < min_edge else ("Home" if edge_points > 0 else "Away")

    confidence_score = min(100, round((abs_edge / 7.5) * 100, 1))

    if abs_edge >= 5.0:
        stars = "⭐⭐⭐⭐⭐ (Max Play)"
    elif abs_edge >= 3.5:
        stars = "⭐⭐⭐⭐ (Strong)"
    elif abs_edge >= 2.0:
        stars = "⭐⭐⭐ (Standard)"
    elif abs_edge >= 1.0:
        stars = "⭐⭐ (Lean)"
    else:
        stars = "Pass"

    return {"edge_points": round(edge_points, 2), "bet": bet,
            "confidence_score": confidence_score, "stars": stars}

def backtest_model(optimal_params, game_outcomes_df, stints_df, odds_dict, min_edge=2.5):
    tracker = PlayerRatingTracker(config=optimal_params, weights=DYNAMIC_WEIGHTS)
    predictions = []
    team_last_game = {}

    for idx, game in tqdm(game_outcomes_df.iterrows(), total=len(game_outcomes_df), desc="Backtesting ATS"):
        gid = game['GAME_ID']
        home_abbr, away_abbr = game['home_team'], game['away_team']
        game_date = game['game_date']

        # Get stints and players
        stints_for_game = stints_df[stints_df['GAME_ID'] == gid]
        all_players = set()
        for _, st in stints_for_game.iterrows():
            all_players.update(st['HOME_players'].split('-'))
            all_players.update(st['AWAY_players'].split('-'))
        all_players = [p for p in all_players if p and p != 'nan']

        # Apply inactivity decay before ratings are used
        tracker.apply_inactivity_decay(all_players, game_date)

        # Average Elo ratings
        def get_avg_elo(players, which):
            ratings = [tracker.get_player(p)[which] for p in players]
            return np.mean(ratings) if ratings else 1500

        home_players = [p for p in all_players if any(p in st.split('-') for st in stints_for_game['HOME_players'])]
        away_players = [p for p in all_players if any(p in st.split('-') for st in stints_for_game['AWAY_players'])]

        home_off = get_avg_elo(home_players, 'O_Elo')
        home_def = get_avg_elo(home_players, 'D_Elo')
        away_off = get_avg_elo(away_players, 'O_Elo')
        away_def = get_avg_elo(away_players, 'D_Elo')

        # --- Schedule & altitude adjustments (realistic) ---
        def rest_penalty(team, date):
            if team not in team_last_game:
                return 0
            days = (date - team_last_game[team]).days
            return -5 if days == 1 else 0   # -5 Elo ≈ -0.5 points

        home_off += rest_penalty(home_abbr, game_date) + (5 if home_abbr in ['DEN','UTA'] else 0)
        away_off += rest_penalty(away_abbr, game_date)

        team_last_game[home_abbr] = game_date
        team_last_game[away_abbr] = game_date

        # --- Model spread ---
        pred_h_ppp = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] + optimal_params.get('HOME_PPP_BOOST', 0.0) + \
                     (home_off - away_def) / optimal_params['ELO_SCALING_FACTOR']
        pred_a_ppp = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] - optimal_params.get('HOME_PPP_BOOST', 0.0) + \
                     (away_off - home_def) / optimal_params['ELO_SCALING_FACTOR']
        model_spread = (pred_h_ppp - pred_a_ppp) * 100.0

        market_spread, _ = get_odds_for_game(home_abbr, away_abbr, odds_dict)
        analysis = calculate_bet_confidence(model_spread, market_spread, min_edge=min_edge)

        actual_margin = game['home_score'] - game['away_score']
        ats_result = "N/A"
        if market_spread is not None and analysis['bet'] != "Pass":
            ats_margin = actual_margin + market_spread
            if ats_margin == 0:
                ats_result = "Push"
            elif (analysis['bet'] == "Home" and ats_margin > 0) or (analysis['bet'] == "Away" and ats_margin < 0):
                ats_result = "Win"
            else:
                ats_result = "Loss"

        predictions.append({
            'Game_ID': gid, 'Home': home_abbr, 'Away': away_abbr,
            'Model_Spread': round(model_spread, 2), 'Market_Spread': market_spread,
            'Spread_Error': abs(model_spread - actual_margin),
            'ATS_Bet': analysis['bet'], 'ATS_Edge': analysis['edge_points'],
            'Confidence': analysis['confidence_score'], 'Stars': analysis['stars'],
            'ATS_Result': ats_result
        })

        # Update Elo after the game (use process_stint)
        season_progress = idx / len(game_outcomes_df)
        for _, st in stints_for_game.iterrows():
            tracker.process_stint(
                ids_A=st['HOME_players'].split('-'),
                ids_B=st['AWAY_players'].split('-'),
                poss=st['possessions'],
                xpts_A=st['home_xpts'],
                xpts_B=st['away_xpts'],
                usage_A=st['home_usage'],
                usage_B=st['away_usage'],
                period=st['PERIOD'],
                start_score_A=st['HOME_SCORE_START'],
                start_score_B=st['AWAY_SCORE_START'],
                end_score_A=st['HOME_SCORE_END'],
                end_score_B=st['AWAY_SCORE_END'],
                season_progress=season_progress
            )

    pred_df = pd.DataFrame(predictions)
    actionable = pred_df[pred_df['ATS_Result'].isin(['Win', 'Loss'])]
    wins = len(actionable[actionable['ATS_Result'] == 'Win'])
    losses = len(actionable[actionable['ATS_Result'] == 'Loss'])
    ats_pct = wins / (wins + losses) if (wins+losses) > 0 else 0

    print("\n=== AGAINST THE SPREAD (ATS) RESULTS ===")
    print(f"Total games           : {len(pred_df)}")
    print(f"Actionable bets       : {len(actionable)} (edge >= {min_edge} pts)")
    print(f"ATS Record            : {wins} - {losses}")
    print(f"ATS Win %             : {ats_pct:.2%}")
    print(f"Break-even needed     : 52.38% (to beat vig)")
    print(f"Spread MAE            : {pred_df['Spread_Error'].mean():.2f} points")

    return pred_df, tracker

print("\nRunning ATS backtest...")
backtest_df, updated_tracker = backtest_model(optimal_params, game_outcomes_df, stints_df, odds_dict)


Running ATS backtest...


Backtesting ATS:   0%|          | 0/1307 [00:00<?, ?it/s]


=== AGAINST THE SPREAD (ATS) RESULTS ===
Total games           : 1307
Actionable bets       : 1012 (edge >= 2.5 pts)
ATS Record            : 615 - 397
ATS Win %             : 60.77%
Break-even needed     : 52.38% (to beat vig)
Spread MAE            : 10.86 points


In [137]:
# ---------------------------------------------------------
# 10b. ATS Evaluation & Point Edge Confidence Tiers
# ---------------------------------------------------------
import pandas as pd
import numpy as np

print("Evaluating ATS Performance by Raw Point Edge Confidence Tiers...")

# 1. Filter out games where we didn't make a bet or market odds are missing
eval_df = backtest_df.dropna(subset=['Market_Spread']).copy()
eval_df = eval_df[eval_df['ATS_Result'] != 'N/A'].copy()

# 2. Extract Edge and Raw Confidence from the simulation logs
# Positive Edge = Model leans Home | Negative Edge = Model leans Away
eval_df['EDGE'] = eval_df['ATS_Edge']
eval_df['CONFIDENCE'] = eval_df['EDGE'].abs()

# 3. Standardize bet results to uppercase to match grading logic
eval_df['BET_RESULT'] = eval_df['ATS_Result'].str.upper()

# 4. Analyze Performance by Raw Point Edge Tiers
confidence_tiers = [0.0, 1.5, 3.0, 4.5, 6.0, 8.0]
tier_results = []

for threshold in confidence_tiers:
    # Filter games meeting this absolute point edge threshold
    tier_df = eval_df[eval_df['CONFIDENCE'] >= threshold]

    wins = len(tier_df[tier_df['BET_RESULT'] == 'WIN'])
    losses = len(tier_df[tier_df['BET_RESULT'] == 'LOSS'])
    pushes = len(tier_df[tier_df['BET_RESULT'] == 'PUSH'])
    total_bets = wins + losses

    win_pct = wins / total_bets if total_bets > 0 else 0

    # Calculate hypothetical ROI (assuming standard -110 juice)
    # Win pays +0.909 units, Loss loses -1.0 units
    units_won = (wins * 0.909) - losses
    roi = units_won / total_bets if total_bets > 0 else 0

    tier_results.append({
        'Min_Edge (Confidence)': f"{threshold}+ points",
        'Total_Bets': total_bets,
        'Wins': wins,
        'Losses': losses,
        'Pushes': pushes,
        'Win_%': f"{win_pct:.1%}",
        'Units_Profit': round(units_won, 2),
        'ROI': f"{roi:.1%}"
    })

performance_df = pd.DataFrame(tier_results)

print("\n=================================================")
print("🎯 MODEL PERFORMANCE BY POINT EDGE TIER 🎯")
print("=================================================")
display(performance_df)

print("\n🔍 Detailed Look at Top 10 Highest Confidence Bets:")
top_bets = eval_df.sort_values(by='CONFIDENCE', ascending=False).head(10)
display_cols = ['Game_ID', 'Home', 'Away', 'Market_Spread', 'Model_Spread', 'EDGE', 'CONFIDENCE', 'BET_RESULT']
display(top_bets[display_cols])

Evaluating ATS Performance by Raw Point Edge Confidence Tiers...

🎯 MODEL PERFORMANCE BY POINT EDGE TIER 🎯


,Min_Edge (Confidence),Total_Bets,Wins,Losses,Pushes,Win_%,Units_Profit,ROI
0,0.0+ points,1012,615,397,21,60.8%,162.03,16.0%
1,1.5+ points,1012,615,397,21,60.8%,162.03,16.0%
2,3.0+ points,969,591,378,20,61.0%,159.22,16.4%
3,4.5+ points,866,534,332,18,61.7%,153.41,17.7%
4,6.0+ points,753,469,284,14,62.3%,142.32,18.9%
5,8.0+ points,588,376,212,11,63.9%,129.78,22.1%



🔍 Detailed Look at Top 10 Highest Confidence Bets:


,Game_ID,Home,Away,Market_Spread,Model_Spread,EDGE,CONFIDENCE,BET_RESULT
418,22500397,BKN,TOR,23.5,5.15,28.65,28.65,WIN
353,22500363,UTA,OKC,24.0,4.55,28.55,28.55,LOSS
156,22500207,BKN,TOR,23.5,3.99,27.49,27.49,WIN
1035,22500651,MEM,DEN,23.0,4.24,27.24,27.24,WIN
226,22500055,UTA,OKC,24.0,3.19,27.19,27.19,WIN
259,22500287,MEM,DEN,23.0,3.79,26.79,26.79,WIN
326,22500334,WAS,BOS,20.0,3.62,23.62,23.62,WIN
315,22500331,MIL,DET,20.5,2.99,23.49,23.49,WIN
239,22500272,MIL,DET,20.5,2.84,23.34,23.34,WIN
911,22500889,CHI,OKC,19.5,3.82,23.32,23.32,WIN


In [140]:
# -------------------------------
# 11. Live Game Predictor (With Manual Override)
# -------------------------------
def predict_game(home_team_full, away_team_full, game_date, tracker, stints_df, game_outcomes_df,
                 odds_dict, name_to_id, optimal_params, use_real_injuries=True,
                 manual_home_injuries=None, manual_away_injuries=None):
    """Predict a single future game, with a manual override for broken APIs."""
    home_abbr = team_name_to_abbr.get(home_team_full)
    away_abbr = team_name_to_abbr.get(away_team_full)
    if not home_abbr or not away_abbr:
        raise ValueError(f"Unknown team: {home_team_full} or {away_team_full}")

    # Initialize manual lists if None
    manual_home_injuries = manual_home_injuries or []
    manual_away_injuries = manual_away_injuries or []

    # Fetch injuries from API
    api_home_inj_names = []
    api_away_inj_names = []
    if use_real_injuries:
        api_home_inj_names = fetch_injuries(home_abbr, game_date, use_real=True)
        api_away_inj_names = fetch_injuries(away_abbr, game_date, use_real=True)

    # Merge API results with manual overrides (removes duplicates)
    final_home_inj_names = list(set(api_home_inj_names + manual_home_injuries))
    final_away_inj_names = list(set(api_away_inj_names + manual_away_injuries))

    home_inj_ids = [name_to_id.get(n) for n in final_home_inj_names if name_to_id.get(n)]
    away_inj_ids = [name_to_id.get(n) for n in final_away_inj_names if name_to_id.get(n)]

    # Get last known roster (most recent game for that team)
    def get_recent_roster(team_abbr):
        team_games = game_outcomes_df[(game_outcomes_df['home_team'] == team_abbr) |
                                     (game_outcomes_df['away_team'] == team_abbr)]
        if team_games.empty: return []

        last_gid = team_games.iloc[-1]['GAME_ID']
        is_home = team_games.iloc[-1]['home_team'] == team_abbr
        roster = set()
        for _, row in stints_df[stints_df['GAME_ID'] == last_gid].iterrows():
            col = 'HOME_players' if is_home else 'AWAY_players'
            roster.update(row[col].split('-'))
        return [p for p in roster if p and p != 'nan']

    home_players = [p for p in get_recent_roster(home_abbr) if int(p) not in home_inj_ids]
    away_players = [p for p in get_recent_roster(away_abbr) if int(p) not in away_inj_ids]

    if not home_players or not away_players:
        raise ValueError(f"Could not determine active players for {home_abbr} or {away_abbr}")

    # Average Elo ratings
    home_off = np.mean([tracker.get_player(p)['O_Elo'] for p in home_players])
    home_def = np.mean([tracker.get_player(p)['D_Elo'] for p in home_players])
    away_off = np.mean([tracker.get_player(p)['O_Elo'] for p in away_players])
    away_def = np.mean([tracker.get_player(p)['D_Elo'] for p in away_players])

    # Predict
    pred_h_ppp = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] + optimal_params['HOME_PPP_BOOST'] + \
                 (home_off - away_def) / optimal_params['ELO_SCALING_FACTOR']
    pred_a_ppp = DYNAMIC_WEIGHTS['LEAGUE_XPPP'] - optimal_params['HOME_PPP_BOOST'] + \
                 (away_off - home_def) / optimal_params['ELO_SCALING_FACTOR']
    model_spread = (pred_h_ppp - pred_a_ppp) * 100.0
    model_win_prob = 1 / (1 + 10 ** (-model_spread / 15.0))

    # Market odds
    market_spread, market_ml = get_odds_for_game(home_abbr, away_abbr, odds_dict)

    # Confidence analysis
    bet_analysis = calculate_bet_confidence(model_spread, market_spread)

    return {
        'Game': f"{away_team_full} @ {home_team_full}",
        'Vegas_Line': f"{home_abbr} {market_spread}" if market_spread else "No line",
        'Model_Spread': f"{home_abbr} by {round(model_spread, 2)}" if model_spread > 0 else f"{away_abbr} by {round(abs(model_spread), 2)}",
        'Model_Win_Prob': f"{round(model_win_prob * 100, 1)}% (home win)",
        'Spread_Edge': f"{bet_analysis['edge_points']:+.2f} points",
        'Recommendation': f"BET {bet_analysis['bet']} ATS" if bet_analysis['bet'] != "Pass" else "PASS",
        'Confidence_Score': f"{bet_analysis['confidence_score']}/100",
        'Star_Rating': bet_analysis['stars'],
        'Excluded_Injuries': f"{len(home_inj_ids)} home / {len(away_inj_ids)} away",
        'Injured_Names': f"H: {', '.join(final_home_inj_names) if final_home_inj_names else 'None'} | A: {', '.join(final_away_inj_names) if final_away_inj_names else 'None'}"
    }

In [141]:
# -------------------------------
# 12. Example - Live Prediction with Manual Override
# -------------------------------
# Let's test the Lakers vs Warriors game assuming Steph Curry and Draymond Green are out.
future = predict_game(
    home_team_full="Los Angeles Lakers",
    away_team_full="Golden State Warriors",
    game_date="2026-05-25",
    tracker=updated_tracker,
    stints_df=stints_df,
    game_outcomes_df=game_outcomes_df,
    odds_dict=odds_dict,
    name_to_id=name_to_id,
    optimal_params=optimal_params,
    use_real_injuries=True,
    manual_home_injuries=[], # Put Lakers scratches here (e.g., ['LeBron James'])
    manual_away_injuries=['Stephen Curry', 'Draymond Green'] # Put Warriors scratches here
)

print("\n" + "="*60)
print("🏀 LIVE PREDICTION REPORT 🏀")
print("="*60)
for key, value in future.items():
    print(f"{key.ljust(20)} : {value}")
print("="*60)

⚠️ Injury fetch failed for LAL on 2026-05-25: can't compare datetime.datetime to datetime.date
⚠️ Injury fetch failed for GSW on 2026-05-25: can't compare datetime.datetime to datetime.date

🏀 LIVE PREDICTION REPORT 🏀
Game                 : Golden State Warriors @ Los Angeles Lakers
Vegas_Line           : LAL 2.5
Model_Spread         : LAL by 4.05
Model_Win_Prob       : 65.1% (home win)
Spread_Edge          : +6.55 points
Recommendation       : BET Home ATS
Confidence_Score     : 87.4/100
Star_Rating          : ⭐⭐⭐⭐⭐ (Max Play)
Excluded_Injuries    : 0 home / 2 away
Injured_Names        : H: None | A: Stephen Curry, Draymond Green
